# Managing projections with GeoMakie

## The packages
We will rely essentialy on the plotting library [Makie](https://makie.org/website/) and on [GeoMakie](https://geo.makie.org/stable/), to display information on a given map projection.    
Additionaly we will load [NCDatasets](https://github.com/JuliaGeo/NCDatasets.jl) to load some data.        

There are several backends. For instance:
- if you want to work OpenGL, then you need to load `GLMakie`
- if you want to render 2D by Cairo Graphics, you will use `CairoMakie`.

In [1]:
using Pkg
Pkg.activate("../")
Pkg.instantiate()
using Makie
using CairoMakie
using GeoMakie
using NCDatasets
using JupyterFormatter
enable_autoformat()

  Activating project at `~/Projects/maps-with-makie`
┌ Warning: The project dependencies or compat requirements have changed since the manifest was last resolved.
│ It is recommended to `Pkg.resolve()` or consider `Pkg.update()` if necessary.
└ @ Pkg.API ~/.julia/juliaup/julia-1.13.2+0.x64.linux.gnu/share/julia/stdlib/v1.13/Pkg/src/API.jl:1312
Precompiling packages...
   3.4 s  ✓ s2n_tls_jll
   3.5 s  ✓ Libuuid_jll
   3.7 s  ✓ Libmount_jll
   3.7 s  ✓ libaom_jll
   3.9 s  ✓ FriBidi_jll
   3.9 s  ✓ Expat_jll
   4.0 s  ✓ libzip_jll
   4.1 s  ✓ FreeType2_jll
   6.4 s  ✓ CommonDataModel
   2.9 s  ✓ IntervalArithmetic → IntervalArithmeticIrrationalConstantsExt
   4.5 s  ✓ OpenEXR_jll
   4.1 s  ✓ aws_c_io_jll
   4.5 s  ✓ Xorg_libX11_jll
   4.3 s  ✓ FreeType
   6.6 s  ✓ MPICH_jll
   6.3 s  ✓ Glib_jll
  11.9 s  ✓ Roots
   4.2 s  ✓ aws_c_http_jll
   6.4 s  ✓ Fontconfig_jll
   4.4 s  ✓ Xorg_libXfixes_jll
   4.0 s  ✓ Xorg_libXrender_jll
   5.4 s  ✓ OpenEXR
   2.5 s  ✓ Roots → RootsChainRulesCoreE

   4.4 s  ✓ OpenEXR
   5.2 s  ✓ Glib_jll
   5.0 s  ✓ Fontconfig_jll
   2.4 s  ✓ libva_jll
   9.5 s  ✓ SpecialFunctions
  10.1 s  ✓ Roots
   4.1 s  ✓ Libglvnd_jll
   1.8 s  ✓ ColorVectorSpace → SpecialFunctionsExt
   1.7 s  ✓ Roots → RootsChainRulesCoreExt
   2.1 s  ✓ Roots → RootsUnitfulExt
   3.6 s  ✓ Cairo_jll
   3.8 s  ✓ StatsFuns
   2.0 s  ✓ HarfBuzz_jll
   5.1 s  ✓ SpecialFunctions → SpecialFunctionsChainRulesCoreExt
   1.3 s  ✓ StatsFuns → StatsFunsInverseFunctionsExt
   4.0 s  ✓ libwebp_jll
   2.9 s  ✓ StatsFuns → StatsFunsChainRulesCoreExt
   2.7 s  ✓ libass_jll
   3.7 s  ✓ WebP
  20.8 s  ✓ SIMD
  20.9 s  ✓ StaticArrays
  21.1 s  ✓ JSON
   4.7 s  ✓ FFMPEG_jll
   1.7 s  ✓ Accessors → StaticArraysExt
   1.8 s  ✓ StructArrays → StructArraysStaticArraysExt
   1.9 s  ✓ FillArrays → FillArraysStaticArraysExt
   2.1 s  ✓ StaticArrays → StaticArraysStatisticsExt
   2.2 s  ✓ StaticArrays → StaticArraysChainRulesCoreExt
   2.3 s  ✓ ConstructionBase → ConstructionBaseStaticArraysExt
   2.

1-element Vector{Function}:
 format_current_cell (generic function with 1 method)

## Data
We will load the GEBCO bathymetry.

In [7]:
datafile = "gebco_30sec_16.nc"

if isfile(datafile)
    @info("File already downloaded")
else
    download("https://dox.uliege.be/index.php/s/U0pqyXhcQrXjEUX/download", datafile)
end

[ Info: File already downloaded


In order to slightly reduce the matrix size, we will sub-sample the dataset with a factor NN.

In [8]:
NN = 4
ds = NCDataset(datafile)
lon = ds["lon"][1:NN:end]
lat = ds["lat"][1:NN:end]
bathy = ds["bat"][1:NN:end, 1:NN:end]
close(ds)
@info(size(bathy));

[ Info: (675, 338)


## Working with GeoMakie
The procedure for the creation of a plot is very similar, the main difference is that during the creation of the Axis (which is called here `GeoAxis`), one has to define a projection.
### Creating a projection
There are different ways to create a projection. They are defined as _proj-string_ (see https://proj.org/en/stable/operations/projections/index.html).
#### Global projections
We present a few examples using global projections

In [8]:
globalproj1 = "+proj=ortho +lon_0=8.724647 +lat_0=42.583003"
globalproj2 = "+proj=merc"
globalproj3 = "+proj=moll"

"+proj=moll"

In [9]:
function make_map_proj(lon, lat, bathy, projection::String)
    f = Figure(size = (500, 500))
    ax = GeoAxis(f[1, 1], dest = projection, title = "Bathymetry with Makie")
    hm = surface!(
        ax,
        lon,
        lat,
        zeros(size(bathy));
        color = bathy,
        colormap = topocmap,
        colorrange = [-5000, 5000],
        shading = NoShading,
        interpolate = false,
    )
    contour!(ax, lon, lat, bathy, levels = [0.0], color = :black)
    Colorbar(f[1, 2], hm, label = "(m)", labelrotation = 0)
    display(f)
end

make_map_proj (generic function with 1 method)

In [10]:
make_map_proj(lon, lat, bathy, globalproj1)

LoadError: UndefVarError: `lon` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [11]:
make_map_proj(globalproj2)

LoadError: MethodError: no method matching make_map_proj(::String)
The function `make_map_proj` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  make_map_proj(::Any, [91m::Any[39m, [91m::Any[39m, [91m::String[39m)
[0m[90m   @[39m [36mMain[39m [90m[4mIn[9]:1[24m[39m


In [12]:
make_map_proj(globalproj3)

LoadError: MethodError: no method matching make_map_proj(::String)
The function `make_map_proj` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  make_map_proj(::Any, [91m::Any[39m, [91m::Any[39m, [91m::String[39m)
[0m[90m   @[39m [36mMain[39m [90m[4mIn[9]:1[24m[39m


## Local projections
There are plenty of choices for local projections, most of the time it is recommended to avoid Mercator

In [13]:
domain = [0.88429, 4.85484, 38.38061, 40.486754050788335]

4-element Vector{Float64}:
  0.88429
  4.85484
 38.38061
 40.486754050788335

In [14]:
ds = NCDataset(datafile)
lon = ds["lon"][:]
lat = ds["lat"][:]
goodlon = findall((lon .>= domain[1]) .& (lon .<= domain[2]))
goodlat = findall((lat .>= domain[3]) .& (lat .<= domain[4]))

lonregion = lon[goodlon]
latregion = lat[goodlat]
bathyregion = ds["bat"][goodlon, goodlat]
close(ds)
@info(size(bathycalvi));

LoadError: UndefVarError: `NCDataset` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [15]:
heatmap(lonregion, latregion, bathyregion)

LoadError: UndefVarError: `lonregion` not defined in `Main`
Suggestion: check for spelling errors or missing imports.